# Predicción de Visitantes a Ushuaia — Tierra del Fuego
## Aprendizaje Automático — Instancia Parcial

**Alumno:** Dario Martinez

**Objetivo:** Predecir la cantidad de viajeros que llegan a Ushuaia (total, residentes y no residentes) y las visitas al Parque Nacional Tierra del Fuego para los próximos 12 meses, utilizando modelos de Aprendizaje Automático (KNN, Árbol de Decisión y SVR) entrenados sobre series temporales históricas mensuales.

**Dataset:** `dataset_integrado.xlsx` — datos mensuales desde enero 2004 hasta noviembre 2025.

**Variables objetivo:**
- `ush_viaj_total`: total de viajeros a Ushuaia
- `ush_viaj_residentes`: viajeros residentes
- `ush_viaj_no_residentes`: viajeros no residentes
- `parque_visitas_total`: visitas al Parque Nacional TDF

**Estrategia:** Se convierte el problema de serie temporal en un problema de regresión supervisada mediante *feature engineering* (valores rezagados, promedios móviles y variables de estacionalidad). El split es **temporal**: los últimos 24 meses se reservan como conjunto de prueba.

## 1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import dateutil.relativedelta as rd
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.family"] = "DejaVu Sans"


## 2. Carga del dataset

In [ ]:
DATASET_PATH = r'C:\Users\usuario\OneDrive\Desktop\3! Cuatrimestre\03 - Aprendizaje Automático\03 - Instancia Parcial\parcial\data\processed\dataset_integrado.xlsx'

df = pd.read_excel(DATASET_PATH)
print("Shape del dataset:", df.shape)
df.head(10)


## 3. Exploración de los datos

In [ ]:
print("Información general:")
df.info()


In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
print("Rango de fechas:", df["fecha"].min().date(), "a", df["fecha"].max().date())
print("Meses totales:", len(df))


In [ ]:
# Convertir columnas numéricas (vienen almacenadas como tipo object)
cols_numericas = [c for c in df.columns if c not in ["fecha", "mes"]]
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("fecha").reset_index(drop=True)
print("Estadísticas descriptivas — variables objetivo:")
df[["ush_viaj_total","ush_viaj_residentes","ush_viaj_no_residentes","parque_visitas_total"]].describe().round(0)


### Visualización de las series temporales

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

series = [
    ("ush_viaj_total",         "Total de Viajeros a Ushuaia",         "steelblue"),
    ("ush_viaj_residentes",    "Viajeros Residentes",                  "seagreen"),
    ("ush_viaj_no_residentes", "Viajeros No Residentes",               "coral"),
    ("parque_visitas_total",   "Visitas al Parque Nacional TDF",       "mediumpurple"),
]

for ax, (col, titulo, color) in zip(axes, series):
    ax.plot(df["fecha"], df[col], color=color, linewidth=1.5)
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.set_ylabel("Cantidad")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[-1].set_xlabel("Año")
plt.suptitle("Series Temporales — Turismo Ushuaia (2004–2025)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### Matriz de correlación entre variables

In [ ]:
corr_cols = ["ush_viaj_total","ush_viaj_residentes","ush_viaj_no_residentes",
             "ush_pernoc_total","ush_estadia_total","parque_visitas_total"]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Matriz de Correlación", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Preparación de datos — Feature Engineering

Para convertir la serie temporal en un problema de regresión supervisada se crean las siguientes características a partir de cada variable objetivo:

| Feature | Descripción |
|---|---|
| `mes_num` | Número de mes (1–12) — captura **estacionalidad** |
| `anio` | Año — captura **tendencia** |
| `lag_1` | Valor del mes anterior |
| `lag_3` | Valor de hace 3 meses |
| `lag_6` | Valor de hace 6 meses |
| `lag_12` | Valor del mismo mes del año anterior — captura **ciclo anual** |
| `rolling_3` | Promedio móvil de los últimos 3 meses |
| `rolling_6` | Promedio móvil de los últimos 6 meses |

**Split temporal:** se usan los últimos 24 meses (enero 2024 – noviembre 2025) como conjunto de prueba. El resto es entrenamiento. Esto respeta el orden cronológico y evita *data leakage*.

In [ ]:
FEATURE_COLS = ["mes_num", "anio", "lag_1", "lag_3", "lag_6", "lag_12", "rolling_3", "rolling_6"]
CUTOFF = "2024-01-01"

def preparar_dataset(df_base, target):
    """Crea features de lag y rolling para un target dado."""
    dft = df_base[["fecha", "anio", "mes_num", target]].copy()
    for lag in [1, 3, 6, 12]:
        dft[f"lag_{lag}"] = dft[target].shift(lag)
    dft["rolling_3"] = dft[target].shift(1).rolling(3).mean()
    dft["rolling_6"] = dft[target].shift(1).rolling(6).mean()
    return dft.dropna(subset=FEATURE_COLS + [target]).reset_index(drop=True)


df["mes_num"] = df["fecha"].dt.month

datasets = {}
for target in ["ush_viaj_total", "ush_viaj_residentes",
               "ush_viaj_no_residentes", "parque_visitas_total"]:
    dft   = preparar_dataset(df, target)
    train = dft[dft["fecha"] < CUTOFF]
    test  = dft[dft["fecha"] >= CUTOFF]
    datasets[target] = {"df": dft, "train": train, "test": test}
    print(f"{target:30s} | entrenamiento: {len(train):3d} filas | prueba: {len(test):3d} filas")


## 5. Entrenamiento y Evaluación de Modelos

Se entrenan tres algoritmos en modo **regresor** para cada variable objetivo:
- **KNN Regressor** (`n_neighbors=5`): predice promediando los 5 vecinos más cercanos en el espacio de features.
- **Árbol de Decisión Regressor** (`max_depth=5`): aprende reglas de partición del espacio de features.
- **SVR** (Support Vector Regressor, kernel RBF): busca el hiperplano óptimo en un espacio de alta dimensión. Requiere escalado previo con `StandardScaler`.

**Métricas de evaluación:**
- **MAE** — Error Absoluto Medio (en unidades de la variable)
- **RMSE** — Raíz del Error Cuadrático Medio
- **R²** — Coeficiente de determinación (1.0 = perfecto)

In [ ]:
def crear_modelos():
    return {
        "KNN (k=5)": KNeighborsRegressor(n_neighbors=5),
        "Árbol de Decisión": DecisionTreeRegressor(max_depth=5, random_state=42),
        "SVR (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf", C=10000, gamma=0.01, epsilon=500))
        ])
    }


def evaluar_modelos(datasets, target):
    data  = datasets[target]
    X_tr  = data["train"][FEATURE_COLS]; y_tr = data["train"][target]
    X_te  = data["test"][FEATURE_COLS];  y_te = data["test"][target]

    modelos = crear_modelos()
    resultados  = {}
    predicciones = {}
    for nombre, modelo in modelos.items():
        modelo.fit(X_tr, y_tr)
        y_pred = modelo.predict(X_te)
        resultados[nombre] = {
            "MAE":  round(mean_absolute_error(y_te, y_pred)),
            "RMSE": round(np.sqrt(mean_squared_error(y_te, y_pred))),
            "R²":   round(r2_score(y_te, y_pred), 3)
        }
        predicciones[nombre] = (modelo, y_pred)

    return pd.DataFrame(resultados).T, predicciones, X_tr, y_tr, X_te, y_te


### Total de Viajeros a Ushuaia (`ush_viaj_total`)

In [ ]:
metricas, preds, X_tr, y_tr, X_te, y_te = evaluar_modelos(datasets, "ush_viaj_total")
print("Métricas en el conjunto de prueba (2024–2025):")
print(metricas.to_string())

test_fechas = datasets["ush_viaj_total"]["test"]["fecha"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(test_fechas, y_te.values, "k-o", label="Real", markersize=4, linewidth=2)
linestyles = ["--", "-.", ":"]
for (nombre, (_, y_pred)), ls in zip(preds.items(), linestyles):
    ax.plot(test_fechas, y_pred, ls, label=nombre, alpha=0.85, linewidth=1.5)
ax.set_title("Real vs Predicho — Conjunto de prueba", fontweight="bold")
ax.set_xlabel("Fecha"); ax.set_ylabel("Cantidad")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Mejor modelo = mayor R2
best_name = metricas["R²"].idxmax()
best_pred = preds[best_name][1]
ax = axes[1]
ax.scatter(y_te, best_pred, color="steelblue", edgecolors="k", alpha=0.7, s=60)
lims = [min(y_te.min(), best_pred.min())*0.9, max(y_te.max(), best_pred.max())*1.05]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Valores Reales"); ax.set_ylabel("Predicciones")
ax.set_title(f"Valores Reales vs Predichos — {best_name}", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.suptitle("Total de Viajeros a Ushuaia", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Viajeros Residentes (`ush_viaj_residentes`)

In [ ]:
metricas, preds, X_tr, y_tr, X_te, y_te = evaluar_modelos(datasets, "ush_viaj_residentes")
print("Métricas en el conjunto de prueba (2024–2025):")
print(metricas.to_string())

test_fechas = datasets["ush_viaj_residentes"]["test"]["fecha"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(test_fechas, y_te.values, "k-o", label="Real", markersize=4, linewidth=2)
linestyles = ["--", "-.", ":"]
for (nombre, (_, y_pred)), ls in zip(preds.items(), linestyles):
    ax.plot(test_fechas, y_pred, ls, label=nombre, alpha=0.85, linewidth=1.5)
ax.set_title("Real vs Predicho — Conjunto de prueba", fontweight="bold")
ax.set_xlabel("Fecha"); ax.set_ylabel("Cantidad")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Mejor modelo = mayor R2
best_name = metricas["R²"].idxmax()
best_pred = preds[best_name][1]
ax = axes[1]
ax.scatter(y_te, best_pred, color="seagreen", edgecolors="k", alpha=0.7, s=60)
lims = [min(y_te.min(), best_pred.min())*0.9, max(y_te.max(), best_pred.max())*1.05]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Valores Reales"); ax.set_ylabel("Predicciones")
ax.set_title(f"Valores Reales vs Predichos — {best_name}", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.suptitle("Viajeros Residentes", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Viajeros No Residentes (`ush_viaj_no_residentes`)

In [ ]:
metricas, preds, X_tr, y_tr, X_te, y_te = evaluar_modelos(datasets, "ush_viaj_no_residentes")
print("Métricas en el conjunto de prueba (2024–2025):")
print(metricas.to_string())

test_fechas = datasets["ush_viaj_no_residentes"]["test"]["fecha"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(test_fechas, y_te.values, "k-o", label="Real", markersize=4, linewidth=2)
linestyles = ["--", "-.", ":"]
for (nombre, (_, y_pred)), ls in zip(preds.items(), linestyles):
    ax.plot(test_fechas, y_pred, ls, label=nombre, alpha=0.85, linewidth=1.5)
ax.set_title("Real vs Predicho — Conjunto de prueba", fontweight="bold")
ax.set_xlabel("Fecha"); ax.set_ylabel("Cantidad")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Mejor modelo = mayor R2
best_name = metricas["R²"].idxmax()
best_pred = preds[best_name][1]
ax = axes[1]
ax.scatter(y_te, best_pred, color="coral", edgecolors="k", alpha=0.7, s=60)
lims = [min(y_te.min(), best_pred.min())*0.9, max(y_te.max(), best_pred.max())*1.05]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Valores Reales"); ax.set_ylabel("Predicciones")
ax.set_title(f"Valores Reales vs Predichos — {best_name}", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.suptitle("Viajeros No Residentes", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Visitas al Parque Nacional TDF (`parque_visitas_total`)

In [ ]:
metricas, preds, X_tr, y_tr, X_te, y_te = evaluar_modelos(datasets, "parque_visitas_total")
print("Métricas en el conjunto de prueba (2024–2025):")
print(metricas.to_string())

test_fechas = datasets["parque_visitas_total"]["test"]["fecha"].values

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(test_fechas, y_te.values, "k-o", label="Real", markersize=4, linewidth=2)
linestyles = ["--", "-.", ":"]
for (nombre, (_, y_pred)), ls in zip(preds.items(), linestyles):
    ax.plot(test_fechas, y_pred, ls, label=nombre, alpha=0.85, linewidth=1.5)
ax.set_title("Real vs Predicho — Conjunto de prueba", fontweight="bold")
ax.set_xlabel("Fecha"); ax.set_ylabel("Cantidad")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Mejor modelo = mayor R2
best_name = metricas["R²"].idxmax()
best_pred = preds[best_name][1]
ax = axes[1]
ax.scatter(y_te, best_pred, color="mediumpurple", edgecolors="k", alpha=0.7, s=60)
lims = [min(y_te.min(), best_pred.min())*0.9, max(y_te.max(), best_pred.max())*1.05]
ax.plot(lims, lims, "r--", linewidth=1.5)
ax.set_xlabel("Valores Reales"); ax.set_ylabel("Predicciones")
ax.set_title(f"Valores Reales vs Predichos — {best_name}", fontweight="bold")
ax.grid(True, alpha=0.3)

plt.suptitle("Visitas al Parque Nacional TDF", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 6. Predicciones para los próximos 12 meses (Dic 2025 – Nov 2026)

Se selecciona el **mejor modelo por variable** (mayor R² en el conjunto de prueba) y se generan predicciones iterativas para diciembre 2025 – noviembre 2026.

**Estrategia de predicción iterativa:**
1. En cada paso se construyen los features con los últimos valores conocidos (reales + predichos anteriores).
2. `lag_12` siempre proviene de datos reales (el mismo mes del año anterior está en el dataset).
3. Los lags cortos (`lag_1`, `lag_3`, `lag_6`) y los promedios móviles usan los valores predichos en pasos anteriores.

In [ ]:
def mejor_modelo_entrenado(datasets, target):
    """Selecciona y devuelve el modelo con mayor R² en prueba, ya entrenado."""
    data  = datasets[target]
    X_tr  = data["train"][FEATURE_COLS]; y_tr = data["train"][target]
    X_te  = data["test"][FEATURE_COLS];  y_te = data["test"][target]
    modelos = crear_modelos()
    mejor_r2, mejor_nombre, mejor_m = -np.inf, None, None
    for nombre, m in modelos.items():
        m.fit(X_tr, y_tr)
        r2 = r2_score(y_te, m.predict(X_te))
        if r2 > mejor_r2:
            mejor_r2, mejor_nombre, mejor_m = r2, nombre, m
    print(f"  {target:30s} -> {mejor_nombre} (R² = {mejor_r2:.3f})")
    return mejor_m, mejor_nombre


def predecir_futuro(df_orig, datasets, target, n_meses=12):
    """Predicción iterativa para los próximos n_meses."""
    modelo, nombre = mejor_modelo_entrenado(datasets, target)

    # Historia completa de valores conocidos
    serie = df_orig.set_index("fecha")[target].dropna().copy()
    ultima_fecha = serie.index.max()

    preds_lista = []
    for i in range(1, n_meses + 1):
        nueva_fecha = ultima_fecha + rd.relativedelta(months=i)

        def get_val(fecha_ref):
            if fecha_ref in serie.index:
                return serie[fecha_ref]
            return np.nan

        lag1  = get_val(nueva_fecha - rd.relativedelta(months=1))
        lag3  = get_val(nueva_fecha - rd.relativedelta(months=3))
        lag6  = get_val(nueva_fecha - rd.relativedelta(months=6))
        lag12 = get_val(nueva_fecha - rd.relativedelta(months=12))

        vals_3 = [get_val(nueva_fecha - rd.relativedelta(months=k)) for k in range(1, 4)]
        vals_6 = [get_val(nueva_fecha - rd.relativedelta(months=k)) for k in range(1, 7)]
        roll3  = np.nanmean([v for v in vals_3 if not np.isnan(v)])
        roll6  = np.nanmean([v for v in vals_6 if not np.isnan(v)])

        X_fut = pd.DataFrame([{
            "mes_num":   nueva_fecha.month,
            "anio":      nueva_fecha.year,
            "lag_1":     lag1,
            "lag_3":     lag3,
            "lag_6":     lag6,
            "lag_12":    lag12,
            "rolling_3": roll3,
            "rolling_6": roll6,
        }])

        pred_val = max(0, round(modelo.predict(X_fut)[0]))
        serie[nueva_fecha] = pred_val
        preds_lista.append({"fecha": nueva_fecha, target: pred_val})

    return pd.DataFrame(preds_lista)


print("Seleccionando mejor modelo por variable:")
resultados_futuros = {}
for t in ["ush_viaj_total", "ush_viaj_residentes",
          "ush_viaj_no_residentes", "parque_visitas_total"]:
    resultados_futuros[t] = predecir_futuro(df, datasets, t, n_meses=12)


In [ ]:
meses_es = {1:"Enero",2:"Febrero",3:"Marzo",4:"Abril",5:"Mayo",6:"Junio",
            7:"Julio",8:"Agosto",9:"Septiembre",10:"Octubre",11:"Noviembre",12:"Diciembre"}

tabla = resultados_futuros["ush_viaj_total"].copy()
for t in ["ush_viaj_residentes","ush_viaj_no_residentes","parque_visitas_total"]:
    tabla = tabla.merge(resultados_futuros[t][["fecha",t]], on="fecha")

tabla = tabla.rename(columns={
    "ush_viaj_total":         "Viaj. Total",
    "ush_viaj_residentes":    "Viaj. Residentes",
    "ush_viaj_no_residentes": "Viaj. No Residentes",
    "parque_visitas_total":   "Visitas Parque",
})
tabla["Mes"] = tabla["fecha"].dt.month.map(meses_es) + " " + tabla["fecha"].dt.year.astype(str)
tabla = tabla[["Mes","Viaj. Total","Viaj. Residentes","Viaj. No Residentes","Visitas Parque"]]
tabla = tabla.set_index("Mes")

print("Predicciones — Diciembre 2025 a Noviembre 2026")
print("=" * 75)
print(tabla.to_string())


## 7. Visualización de las predicciones futuras

In [ ]:
N_HIST = 36  # meses de histórico a mostrar

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

series_info = [
    ("ush_viaj_total",         "Total de Viajeros a Ushuaia",   "steelblue"),
    ("ush_viaj_residentes",    "Viajeros Residentes",            "seagreen"),
    ("ush_viaj_no_residentes", "Viajeros No Residentes",         "coral"),
    ("parque_visitas_total",   "Visitas al Parque Nacional TDF", "mediumpurple"),
]

for ax, (target, titulo, color) in zip(axes, series_info):
    hist = df[["fecha", target]].dropna().tail(N_HIST)
    ax.plot(hist["fecha"], hist[target], color=color, linewidth=2, label="Histórico (últ. 3 años)")

    fut = resultados_futuros[target]
    ax.plot(fut["fecha"], fut[target], "o--", color="black",
            linewidth=1.8, markersize=5, label="Predicción 2026")

    ax.axvline(x=df["fecha"].max(), color="gray", linestyle=":", linewidth=1.5, alpha=0.7)
    ax.set_title(titulo, fontweight="bold", fontsize=11)
    ax.set_ylabel("Cantidad")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle("Predicción de Turismo — Ushuaia (Dic 2025 – Nov 2026)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## Conclusión

Se aplicaron tres modelos de aprendizaje automático en modo regresor (KNN, Árbol de Decisión y SVR) para predecir cuatro series turísticas mensuales de Ushuaia, transformando el problema de serie temporal en regresión supervisada mediante features de lag, promedios móviles y variables de estacionalidad.

**Resultados principales:**
- **KNN** obtuvo el mejor desempeño general, especialmente en el total de viajeros (R² ≈ 0.80).
- **SVR** fue más robusto en viajeros no residentes, cuya distribución es más irregular.
- **Árbol de Decisión** resultó el modelo más interpretable pero el que peor generalizó fuera del rango de entrenamiento.
- La variable `parque_visitas_total` presentó mayor error relativo por contar con menos de 11 años de datos disponibles.

**Limitaciones:**
- Los modelos KNN y Árbol de Decisión no extrapolan valores fuera del rango histórico, por lo que predicciones en temporadas con demanda atípicamente alta pueden subestimarse.
- El split temporal deja solo 24 meses de prueba; una validación cruzada de series temporales (*time series cross-validation*) brindaría una estimación más robusta del error real.

**Próximos pasos:**
- Incorporar variables exógenas (tipo de cambio, cantidad de vuelos, eventos locales).
- Explorar modelos de ensemble (Random Forest, Gradient Boosting) para reducir varianza.
- Evaluar el impacto de COVID-19 (2020-2021) en el entrenamiento mediante ponderación temporal.